### 加载数据

In [1]:
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split

In [82]:
# 加载数据
prices = load_sp500_dataset()

# 切片获取 2015 年之后的数据
prices = prices["2000":]

# 转换为线性收益率
X = prices_to_returns(prices)

# 划分训练集和测试集，不洗牌
X_train, X_test = train_test_split(X, test_size=0.15, shuffle=False)

In [44]:
X_test.shape

(302, 20)

### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

In [4]:
from skfolio.pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated

In [37]:
select_cols = [prices.columns]
for selector in [SelectKExtremes(k=5,),
                 DropCorrelated(threshold=0.4),
                 SelectNonDominated(min_n_assets=5)]:
    selector.fit(X_train)
    names = selector.get_feature_names_out(input_features=prices.columns)
    print(names)
    select_cols.append(names)

['AAPL' 'AMD' 'BBY' 'MSFT' 'PG']
['AMD' 'GE' 'RRC' 'WMT']
['AAPL' 'AMD' 'HD' 'JNJ' 'MSFT' 'PG' 'WMT']


### EqualWeighted基准

In [6]:
from skfolio.optimization import EqualWeighted
from skfolio import Population
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure

In [38]:
ew0 = EqualWeighted(portfolio_params=dict(name="All")).fit(X_train)
ew1 = EqualWeighted(portfolio_params=dict(name="5 Extremes")).fit(X_train[select_cols[1]])
ew2 = EqualWeighted(portfolio_params=dict(name="Correlated <= 0.5")).fit(X_train[select_cols[2]])
ew3 = EqualWeighted(portfolio_params=dict(name="Non Dominated")).fit(X_train[select_cols[3]])

In [39]:
population_train = Population([m.predict(X_train[cols]) for m, cols in zip([ew0, ew1, ew2, ew3],select_cols)])
population_test = Population([m.predict(X_test[cols]) for m, cols in zip([ew0, ew1, ew2, ew3],select_cols)])

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population = population_train + population_test

In [40]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
```
purged_size=0：训练结束与测试开始无缝衔接。
purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。例如，purged_size=1 意味着当前时期的决策从下一个时期才开始影响性能。
建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。
```
- 训练集扩展与尾部数据处理
```
expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。
```

In [ ]:

from skfolio.model_selection import WalkForward

Fold 0:
  Train: index=[0 1]
  Test:  index=[2]
Fold 1:
  Train: index=[1 2]
  Test:  index=[3]
Fold 2:
  Train: index=[2 3]
  Test:  index=[4]
Fold 3:
  Train: index=[3 4]
  Test:  index=[5]


In [91]:
train_portfolios = []
test_portfolios = []
cv = WalkForward(test_size=63, train_size=504, purged_size=1)
for i, (train_index, test_index) in enumerate(cv.split(X)):
    #selector = DropCorrelated(threshold=0.4).fit(X.iloc[train_index])
    selector = SelectNonDominated(min_n_assets=5).fit(X.iloc[train_index])
    selected_cols = selector.get_feature_names_out(input_features=prices.columns)
    X_train = X.iloc[train_index][selected_cols]
    X_test = X.iloc[test_index][selected_cols]
    m = EqualWeighted(portfolio_params=dict(name="Fold %d"%i)).fit(X_train)
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test))

population_train = Population(train_portfolios)
population_test = Population(test_portfolios)

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population = population_train + population_test

In [90]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [92]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)